In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import itertools
from pathlib import Path
import pickle
import re

import matplotlib.pyplot as plt
import mne
from mne.decoding import ReceptiveField
import numpy as np
import pandas as pd
from sklearn.base import clone, BaseEstimator, TransformerMixin, check_is_fitted
from sklearn.model_selection import StratifiedKFold, KFold
import seaborn as sns
from tqdm.auto import tqdm

from src.data import add_metadata_features
from src.stimuli import POD_dict

In [42]:
epochs_path = "outputs/epochs_preprocessed/EC243_epo.fif"
out_path = "outputs/trfs_behavior/EC243/results.pkl"

In [4]:
mne.set_config("MNE_TQDM", "off")

In [ ]:
subject_name = re.findall("(EC[\d]+)_epo", epochs_path)[0]
ep = mne.read_epochs(epochs_path)

In [ ]:
old_metadata = ep.metadata.copy()
ep.metadata = add_metadata_features(old_metadata)

In [46]:
# Prepare design matrix
features_per_phoneme_pair = [
    # ("linear_acoustic_cue", "onset"),
    ("categorical_acoustic_cue", "onset"),

    ("lexical_evidence_cue", "PoD"),

    ("mismatch", "PoD"),
    ("mismatch_left_right", "PoD"),

    # ("behavior_linear", "onset"),
    ("behavior_categorical", "onset"),

    ("feat_behavior_acoustic", "PoD"),
    ("feat_behavior_lexical", "PoD"),
    ("feat_behavior_mismatch_left_right", "PoD"),
]

# Prepare blocks of features to be removed in UV analysis
feature_blocks = {
    "acoustic": [
        # "linear_acoustic_cue",
        "categorical_acoustic_cue",
    ],
    "lexical_evidence": [
        "lexical_evidence_cue",
    ],
    "mismatch": [
        "mismatch",
        "mismatch_left_right",
    ],
    "behavior": [
        # "behavior_linear",
        "behavior_categorical",
    ],
    "behavior_interaction": [
        "feat_behavior_acoustic",
        "feat_behavior_lexical",
        "feat_behavior_mismatch_left_right",
    ]
}

In [ ]:
class TRF(BaseEstimator):
    """
    combines data prep + model fitting in a single estimator
    """
    def __init__(self, estimator, ep: mne.Epochs, features_per_phoneme_pair):
        self.estimator = estimator
        self.ep = ep
        self.features_per_phoneme_pair = features_per_phoneme_pair

        self._update_metadata()

    def _update_metadata(self):
        self.phoneme_pairs = sorted(set(self.ep.metadata.phoneme_pair))
        self.feature_names = [
            f"{feature_name}-{phoneme_pair}"
            for feature_name, _ in self.features_per_phoneme_pair
            for phoneme_pair in self.phoneme_pairs
        ]

    def set_params(self, **params):
        super().set_params(**params)
    
        if "features_per_phoneme_pair" in params or "ep" in params:
            self._update_metadata()

        return self

    def _prepare_design_matrix(self, idxs):
        # HACK this is incorporated here rather than in a Pipeline because we need to
        # modify both X and Y

        X, Y = [], []
        md = self.ep.metadata

        for idx in idxs:
            ep_i = self.ep[idx].get_data().squeeze(0)
            # ep_i: n_channels * n_samples
            md_i = md.iloc[idx]

            # build up design matrix for this trial by column
            Xi = []
            for feature_name, feature_alignment in self.features_per_phoneme_pair:
                for phoneme_pair in self.phoneme_pairs:
                    Xij = np.zeros((ep_i.shape[1], 1))
                    if phoneme_pair == md_i.phoneme_pair:
                        if feature_alignment == "onset":
                            onset_time = 0. - self.ep.tmin
                            onset_sample = int(onset_time * self.ep.info["sfreq"])
                            Xij[onset_sample] = md_i[feature_name]
                        elif feature_alignment == "PoD":
                            PoD_time = md_i.point_of_disambiguation - self.ep.tmin
                            PoD_sample = int(PoD_time * self.ep.info["sfreq"])
                            Xij[PoD_sample] = md_i[feature_name]
                        else:
                            raise ValueError(f"Unknown feature alignment: {feature_alignment}")
                    Xi.append(Xij)
            Xi = np.concatenate(Xi, axis=1)
            X.append(Xi)

            Y.append(ep_i)

        X = np.concatenate(X, axis=0)
        Y = np.concatenate(Y, axis=1).T

        return X, Y

    def fit(self, idxs, y=None):
        X, Y = self._prepare_design_matrix(idxs)

        est = clone(self.estimator)
        est.feature_names = self.feature_names
        self.estimator_ = est.fit(X, Y)

        return self

    def score_multidimensional(self, idxs, y=None):
        check_is_fitted(self)
        X, Y = self._prepare_design_matrix(idxs)

        # returns one score per output channel
        scores = self.estimator_.score(X, Y)
        return scores
    
    def score(self, idxs, y=None):
        check_is_fitted(self)
        scores = self.score_multidimensional(idxs)

        # take mean across electrodes, ignoring negative results
        scores[scores < 0] = np.nan
        if np.isnan(scores).all():
            return 0
        return np.nanmean(scores)

    @property
    def coef_(self):
        return self.estimator_.coef_
    
    def get_feature_names(self, feature_spec):
        feature_name, _ = feature_spec
        return [f"{feature_name}-{phoneme_pair}" for phoneme_pair in self.phoneme_pairs]

    def predict(self, idxs):
        check_is_fitted(self)
        X, _ = self._prepare_design_matrix(idxs)
        return self.estimator_.predict(X)


def estimate_trf_unique_variance(features_per_phoneme_pair,
                                 estimator: TRF, train_idxs, test_idxs):
    overall_score = estimator.score_multidimensional(test_idxs)
    
    scored_blocks, scores = [], []
    for feature_block_name, features in feature_blocks.items():
        estimator_modified = clone(estimator)
        estimator_modified.set_params(
            features_per_phoneme_pair=[f for f in features_per_phoneme_pair if f[0] not in features])
        
        estimator_modified.fit(train_idxs)
        score = estimator_modified.score_multidimensional(test_idxs)
        # ignore subzero score
        score[score < 0] = np.nan

        scored_blocks.append(feature_block_name)
        scores.append(score)

    # electrodes with subzero score should not be considered
    reference_score = overall_score.copy()
    reference_score[reference_score < 0] = np.nan
    score_deltas = reference_score[None, :] - np.array(scores)

    return overall_score, scored_blocks, score_deltas


def estimate_trf(ep: mne.Epochs, features_per_phoneme_pair, num_folds=3):
    Cs = np.logspace(-4, 2, 7)
    md = ep.metadata
    epoch_idxs = md.index.values

    estimator = ReceptiveField(tmin=-0.1, tmax=0.7, sfreq=ep.info["sfreq"])
    outer_cv = StratifiedKFold(num_folds, shuffle=True, random_state=42)
    inner_cv = StratifiedKFold(num_folds, shuffle=True, random_state=42)

    # # DEV
    # epoch_idxs = epoch_idxs[:100]
    # estimator = ReceptiveField(tmin=-0.1, tmax=0.1, sfreq=ep.info["sfreq"])

    pipeline = TRF(estimator, ep, features_per_phoneme_pair)
    param_grid = {"estimator__estimator": Cs}

    from sklearn.model_selection import GridSearchCV, cross_validate
    from sklearn.metrics import r2_score, make_scorer
    clf = GridSearchCV(pipeline, param_grid, cv=inner_cv, n_jobs=1)
    stratify_class = md.loc[epoch_idxs].stratify_class

    cv_results = cross_validate(clf, X=epoch_idxs, y=stratify_class,
                                cv=outer_cv, n_jobs=4,
                                return_estimator=True,
                                return_train_score=True,
                                return_indices=True,
                                verbose=100)

    # Estimate unique variance explained per feature on the outer folds.
    overall_scores, unique_variance_estimates = [], []
    for gs, train_idxs, test_idxs in zip(tqdm(cv_results["estimator"], desc="Estimate unique variance", unit="fold"),
                                         cv_results["indices"]["train"],
                                         cv_results["indices"]["test"]):
        overall_score, block_names, per_block_uv = estimate_trf_unique_variance(
            features_per_phoneme_pair, gs.best_estimator_, train_idxs, test_idxs
        )
        overall_scores.append(overall_score)
        unique_variance_estimates.append(dict(zip(block_names, per_block_uv)))

    overall_scores = pd.DataFrame(overall_scores)
    overall_scores.index.name = "fold"
    overall_scores.columns.name = "electrode"
    unique_variance_estimates = pd.concat({
            fold: pd.DataFrame.from_dict(fold_uv_estimates, orient="index")
            for fold, fold_uv_estimates in enumerate(unique_variance_estimates)
        }, names=["fold", "feature_block"])
    unique_variance_estimates.columns.name = "electrode"
    
    return cv_results, overall_scores, unique_variance_estimates


cv_results, overall_scores, unique_variance_estimates = estimate_trf(ep, features_per_phoneme_pair)


In [ ]:
all_results = {
    "unique_variance_df": unique_variance_estimates,
    "overall_scores_df": overall_scores,

    "features_per_phoneme_pair": features_per_phoneme_pair,
    "feature_blocks": feature_blocks,

    "estimators": [gs.best_estimator_.estimator_ for gs in cv_results["estimator"]],
    "train_score": cv_results["train_score"],
    "test_score": cv_results["test_score"],
}

In [ ]:
with open(out_path, "wb") as f:
    pickle.dump(all_results, f)

## Basic plot test

In [ ]:
# best_uv = all_results["unique_variance_df"].groupby("feature_block").mean().reset_index() \
#     .melt(id_vars="feature_block", var_name="electrode", value_name="unique_variance") \
#     .sort_values("unique_variance", ascending=False)
# best_uv

In [ ]:
# est = all_results["estimators"][0]

In [ ]:
# plot_feature_name = best_uv.iloc[0].feature_name
# plot_electrode = best_uv.iloc[0].electrode

# all_coef = np.array([est.coef_ for est in all_results["estimators"]])
# plot_feature_idxs = [(idx, name) for idx, name in enumerate(all_results["estimators"][0].feature_names)
#                      if name.startswith(plot_feature_name)]
# plot_feature_names = [name for idx, name in plot_feature_idxs]
# plot_feature_idxs = [idx for idx, name in plot_feature_idxs]
# plot_coef = all_coef[:, best_uv.iloc[0].electrode, plot_feature_idxs, :]

# f, ax = plt.subplots(figsize=(10, 6))
# ax.set_title(f"TRF coefficients for {plot_feature_name} on electrode {plot_electrode}")
# palette = sns.color_palette("tab10", n_colors=len(plot_feature_names))
# times = est.delays_ / est.sfreq

# for fold, fold_coefs in enumerate(plot_coef):
#     for (feature_idx, feature), coef in zip(enumerate(plot_feature_names), fold_coefs):
#         ax.plot(times, coef, color=palette[feature_idx], label=feature if fold == 0 else None)

# ax.axvline(0, color="gray", linestyle="--")
# ax.legend()

In [ ]:
# # Visualize feature weighting for top-scoring electrodes
# best_electrodes = np.argsort(np.stack(scores).mean(axis=0))[::-1][:3]
# print(list(zip(best_electrodes, np.stack(scores).mean(axis=0)[best_electrodes])))

# all_coefs = np.stack(coefs).mean(axis=0)
# f, axs = plt.subplots(3, 1, figsize=(10, 3 * 3))
# for ax, electrode_idx in zip(axs, best_electrodes):
#     xticklabels = [f"{delay / model.sfreq:.2f}" if i % 5 == 0 else ""
#                    for i, delay in enumerate(model.delays_)]
#     sns.heatmap(all_coefs[electrode_idx], ax=ax,
#                 xticklabels=xticklabels,
#                 yticklabels=feature_names)
#     ax.set_title(f"Electrode {electrode_idx}")